In [ ]:
#Focus 6831501157-gif Collaborator

In [ ]:
import math
from datetime import datetime, timedelta

# ==========================================
# ฟังก์ชันแปลงเวลาที่กรอก
# ==========================================

def convert_time(value):
    value = value.strip()

    # รองรับ 12:00, 1200, 12000, 830
    if ":" in value:
        parts = value.split(":")
        if len(parts) != 2:
            raise ValueError("รูปแบบเวลาไม่ถูกต้อง")
        hour, minute = parts
    else:
        if len(value) == 5 and value.endswith("0"):
            value = value[:-1]

        if not value.isdigit():
            raise ValueError("กรุณากรอกเวลาเป็นตัวเลข")

        if len(value) <= 2:
            hour, minute = value, "00"
        elif len(value) in (3, 4):
            hour, minute = value[:-2], value[-2:]
        else:
            raise ValueError("รูปแบบเวลาไม่ถูกต้อง")

    hour = int(hour)
    minute = int(minute)

    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError("ชั่วโมงหรือนาทีไม่ถูกต้อง")

    return datetime.strptime(
        f"{hour:02d}:{minute:02d}", "%H:%M"
    )


while True:
    try:
        # ======================================
        # INPUT
        # ======================================

        weight = float(input("น้ำหนักตัว (kg): "))
        age = int(input("อายุ (ปี): "))
        coffees = [
            ("กาแฟดำชงผ่านกระดาษ (Drip)", 240, 95, 96),
            ("กาแฟดำ Drip แก้วใหญ่", 350, 140, 140),
            ("Espresso 1 Shot", 30, 63, 63),
            ("Espresso 2 Shots", 60, 126, 126),
            ("Americano 1 Shot", 240, 63, 63),
            ("Americano 2 Shots", 350, 126, 126),
            ("Latte 1 Shot", 240, 63, 63),
            ("Cappuccino 1 Shot", 180, 63, 63),
            ("Cold Brew", 240, 100, 200),
            ("Instant Coffee", 240, 62, 62),
            ("Decaf Coffee", 240, 1, 5),
        ]
        print("\nเลือกกาแฟ 1 แก้ว/รายการ (ปริมาณคาเฟอีนโดยประมาณ)")
        for number, (name, volume, low, high) in enumerate(coffees, 1):
            amount = str(low) if low == high else f"{low}–{high}"
            print(f"{number:2}. {name} | {volume} ml | {amount} mg")

        choice = int(input("เลือกหมายเลขกาแฟ (1–11): "))
        if not 1 <= choice <= len(coffees):
            raise ValueError("กรุณาเลือกหมายเลขกาแฟ 1–11")
        coffee_name, volume, low, high = coffees[choice - 1]
        dose = high
        if low != high:
            print(f"รายการนี้มีคาเฟอีนประมาณ {low}–{high} mg")
            estimate = input("ใช้ค่า 1 = ต่ำสุด, 2 = สูงสุด (Enter = สูงสุด): ").strip()
            if estimate not in ("", "1", "2"):
                raise ValueError("กรุณาเลือก 1 หรือ 2")
            dose = low if estimate == "1" else high

        drink_time = input("เวลาที่ดื่ม (เช่น 12:00): ")

        if not math.isfinite(weight) or weight <= 0 or age < 0:
            raise ValueError("ข้อมูลไม่ถูกต้อง")

        start_time = convert_time(drink_time)

        # ======================================
        # สูตรที่ 1: Vd
        # ======================================

        # ปริมาตรการกระจายตัว (L)
        Vd = weight * 0.6

        # ======================================
        # สูตรที่ 2: Cmax
        # ======================================

        # ความเข้มข้นเริ่มต้น (mg/L)
        Cmax = dose / Vd

        # ======================================
        # สูตรที่ 3: CL
        # ======================================

        # ค่า CL สมมติตามอายุ
        if age >= 65:
            CL_per_kg = 0.06
        else:
            CL_per_kg = 0.085

        # อัตราการกำจัดรวม (L/h)
        CL = CL_per_kg * weight

        # ======================================
        # สูตรที่ 4: Half-life
        # ======================================

        # ค่าครึ่งชีวิต (ชั่วโมง)
        t_half = math.log(2) * Vd / CL

        # ======================================
        # สูตรที่ 5: ke
        # ======================================

        # ค่าคงที่การกำจัด (1/h)
        ke = math.log(2) / t_half

        # ======================================
        # สูตรที่ 6: เวลาที่ปริมาณคาเฟอีนลดถึงเกณฑ์
        # A(t) = Dose * exp(-ke * t)
        # t = ln(Dose / Target) / ke
        # แบบจำลองนี้ไม่ถึง 0 mg ในเวลาจำกัด จึงใช้ < 1 mg เป็นเกณฑ์ใกล้หมด
        # ======================================

        # ======================================
        # OUTPUT
        # ======================================

        print("\n========== ผลการคำนวณ ==========")
        print(f"น้ำหนัก: {weight:.2f} kg")
        print(f"อายุ: {age} ปี")
        print(f"กาแฟ: {coffee_name} ({volume} ml)")
        print(f"คาเฟอีนเริ่มต้นโดยประมาณ: {dose:.2f} mg")
        print(f"เวลาที่ดื่ม: {start_time:%H:%M} น.")

        print("----------------------------------")
        print(f"Vd = {Vd:.2f} L")
        print(f"Cmax = {Cmax:.2f} mg/L")
        print(f"CL = {CL:.3f} L/h")
        print(f"Half-life = {t_half:.2f} ชั่วโมง")
        print(f"ke = {ke:.4f} 1/h")
        print("----------------------------------")

        print("เวลาแจ้งเตือนโดยประมาณ (แสดงผลใน Notebook)")
        for target, label in ((100, "เหลือไม่เกิน 100 mg"),
                              (50, "เหลือไม่เกิน 50 mg"),
                              (1, "ใกล้หมด (ต่ำกว่า 1 mg)")):
            if target == 1:
                # ปัดไปยังนาทีถัดไป เพื่อให้ต่ำกว่า 1 mg จริง
                total_minutes = max(0, math.floor(math.log(dose / target) / ke * 60) + 1)
            else:
                total_minutes = max(0, math.ceil(math.log(dose / target) / ke * 60))

            end_time = start_time + timedelta(minutes=total_minutes)
            days_later = (end_time.date() - start_time.date()).days
            day_label = "วันเดียวกัน" if days_later == 0 else (
                "วันถัดไป" if days_later == 1 else f"อีก {days_later} วัน")
            if total_minutes == 0:
                print(f"• {label}: ตั้งแต่เวลาดื่ม {start_time:%H:%M} น. (วันเดียวกัน)")
            else:
                print(f"• {label}: {end_time:%H:%M} น. ({day_label}) "
                      f"หลังดื่ม {total_minutes // 60} ชั่วโมง {total_minutes % 60} นาที")

        print("คาเฟอีนไม่ลดถึง 0 mg พอดีตามสมการนี้; ใกล้หมดหมายถึงต่ำกว่า 1 mg")
        print("เป็นค่าประมาณจากแบบจำลองเดิม ไม่ใช่เวลาที่รับรองว่าปลอดคาเฟอีน")
        print("ผลจริงขึ้นอยู่กับแต่ละบุคคลและปริมาณคาเฟอีนจริงในกาแฟ")
        print("==================================")

        break

    except ValueError as error:
        print("ข้อมูลไม่ถูกต้อง:", error)
        print("กรุณากรอกข้อมูลใหม่อีกครั้ง\n")
#focus 